In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ['GOOGLE_API_KEY'] = os.getenv('GOOGLE_API_KEY')
os.environ['HF_TOKEN'] = os.getenv('HUGGINGFACE_API_KEY')
os.environ['TAVILY_API_KEY'] = os.getenv('TAVILY_API_KEY')

In [3]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    # Point to the OpenAI-compatible "door"
    base_url="http://localhost:11434/v1", 
    
    # Required but ignored by Ollama
    api_key="ollama", 
    
    # The exact name you see in 'ollama list'
    model="gemma3:4b", 
    
    temperature=0
)

c:\python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from langchain_google_genai import GoogleGenerativeAI
google_llm = GoogleGenerativeAI(model="gemini-2.5-flash-lite",)

In [5]:
google_llm.invoke("what is weather of delhi today?")

'I don\'t have real-time access to live weather data. My knowledge is based on the data I was trained on, which isn\'t current.\n\nTo get the most accurate and up-to-date weather information for Delhi today, I recommend checking a reliable weather source like:\n\n*   **Weather websites:** AccuWeather, The Weather Channel, Google Weather, India Meteorological Department (IMD)\n*   **Weather apps:** Most smartphones come with a built-in weather app.\n*   **News websites:** Many news organizations provide weather updates.\n\nYou can simply search for "weather Delhi today" on Google or your preferred search engine.'

In [6]:
llm.invoke("What is the capital of France?")

AIMessage(content="The capital of France is **Paris**. \n\nIt's a global center for art, fashion, gastronomy, and culture! 🇫🇷\n", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 16, 'total_tokens': 47, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'gemma3:4b', 'system_fingerprint': 'fp_ollama', 'id': 'chatcmpl-871', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d1e16-ace0-79e2-a528-a593a2a64a91-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 16, 'output_tokens': 31, 'total_tokens': 47, 'input_token_details': {}, 'output_token_details': {}})

In [5]:
from sentence_transformers import SentenceTransformer

Embed_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5293.83it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
class Chatbot:
    def __init__(self,system=""):
        self.system=system
        self.message=[]
        if self.system:
            self.message.append({"role":"system","content":system})
    def __call__(self,message):
        self.message.append({"role":"user","content":message})
        result=self.execute()
        self.message.append({"role":"assistant","content":result})
        return result
        
    def execute(self):
        llm = google_llm
        result = llm.invoke(self.message)
        # Handle both string and message object returns
        if isinstance(result, str):
            return result
        return result.content


In [17]:
bot = Chatbot(system="You are a helpful assistant.")

In [18]:
bot("what is Ocean?")

'The term "ocean" can refer to a few different things, depending on the context:\n\n**1. The Vast Body of Saltwater:**\n\nThis is the most common and widely understood meaning. It refers to the **continuous, interconnected body of saltwater that covers approximately 71% of the Earth\'s surface.** This single, vast ocean is often divided into five major basins for convenience and geographical reference:\n\n*   **Pacific Ocean:** The largest and deepest.\n*   **Atlantic Ocean:** Separates the Americas from Europe and Africa.\n*   **Indian Ocean:** Primarily south of Asia.\n*   **Southern (or Antarctic) Ocean:** Surrounds Antarctica.\n*   **Arctic Ocean:** The smallest and shallowest, located around the North Pole.\n\nThese oceans are crucial for life on Earth, playing vital roles in:\n\n*   **Climate regulation:** They absorb and distribute heat, influencing weather patterns.\n*   **Oxygen production:** Phytoplankton in the ocean produce a significant portion of the Earth\'s oxygen.\n*  

In [19]:
bot.message

[{'role': 'system', 'content': 'You are a helpful assistant.'},
 {'role': 'user', 'content': 'what is Ocean?'},
 {'role': 'assistant',
  'content': 'The term "ocean" can refer to a few different things, depending on the context:\n\n**1. The Vast Body of Saltwater:**\n\nThis is the most common and widely understood meaning. It refers to the **continuous, interconnected body of saltwater that covers approximately 71% of the Earth\'s surface.** This single, vast ocean is often divided into five major basins for convenience and geographical reference:\n\n*   **Pacific Ocean:** The largest and deepest.\n*   **Atlantic Ocean:** Separates the Americas from Europe and Africa.\n*   **Indian Ocean:** Primarily south of Asia.\n*   **Southern (or Antarctic) Ocean:** Surrounds Antarctica.\n*   **Arctic Ocean:** The smallest and shallowest, located around the North Pole.\n\nThese oceans are crucial for life on Earth, playing vital roles in:\n\n*   **Climate regulation:** They absorb and distribute h

In [1]:
prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop your output an Answer.
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.


Your available actions are:
calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point
syntax if necessary

wikipedia:
e.g. wikipedia: Django
Returns a summary from searching Wikipedia

simon_blog_search:
e.g. simon_blog_search: Python
Search Simon's blog for that term

Example session:
Question: What is the capital of France?
Thought: I should look up France on Wikipedia
Action: wikipedia: France
PAUSE

You will be called again with this:
Observation: France is a country. The capital is Paris.

You then output:
Answer: The capital of France is Paris

Please Note: if you get basic conversation questions like "hi","hello","how are you?",\n
you have to answer "hi","hello","i am good".
""".strip()

#### Understand the Regular expression
Pattern Breakdown:

^: This matches the start of a string. It means the string must begin with what follows.
Action:: This is a literal match. It means the string must have the text "Action:" at the beginning.
(\w+):
The parentheses () define a capture group. This allows you to extract part of the string that matches this section.
\w+ matches one or more word characters (letters, digits, and underscores). This will capture a word that follows "Action:".
:: This is a literal colon that separates the word matched by (\w+) from the rest of the string.
(.*):
This is another capture group, where .* matches any character (.) zero or more times (*), which means it captures everything that comes after the second colon.
What does it do?
This regex is looking for a string that:

#### Starts with the text "Action:".
Has a word right after it, separated by a colon.
Then, after another colon, it captures everything that follows.

In [7]:
import re
action_re = re.compile('^Action: (\w+): (.*)')

<>:2: SyntaxWarning: invalid escape sequence '\w'
<>:2: SyntaxWarning: invalid escape sequence '\w'
C:\Users\Manas tiwari\AppData\Local\Temp\ipykernel_15580\952804720.py:2: SyntaxWarning: invalid escape sequence '\w'
  action_re = re.compile('^Action: (\w+): (.*)')


In [8]:
import re

# Compiling the regular expression pattern
action_re = re.compile('^Action: (\w+): (.*)')

# Sample strings
text1 = "Action: Move: North"
text2 = "Action: Jump: High"
text3 = "Error: Not an Action"

<>:4: SyntaxWarning: invalid escape sequence '\w'
<>:4: SyntaxWarning: invalid escape sequence '\w'
C:\Users\Manas tiwari\AppData\Local\Temp\ipykernel_15580\1117371823.py:4: SyntaxWarning: invalid escape sequence '\w'
  action_re = re.compile('^Action: (\w+): (.*)')


In [9]:
# Testing the pattern
match1 = action_re.match(text1)

In [10]:
match1

<re.Match object; span=(0, 19), match='Action: Move: North'>

In [12]:
match1.group(2)

'North'

In [13]:
match3 = action_re.match(text3)
match3

In [14]:
match2 = action_re.match(text2)
match2

<re.Match object; span=(0, 18), match='Action: Jump: High'>

In [15]:
import httpx
def wikipedia(query):
    response = httpx.get("https://en.wikipedia.org/w/api.php", params={
        "action": "query",
        "list": "search",
        "srsearch": query,
        "format": "json"
    })
    return response.json()["query"]["search"][0]["snippet"]


In [16]:
import httpx
def simon_blog_search(query):
    response = httpx.get("https://datasette.simonwillison.net/simonwillisonblog.json", params={
        "sql": """
        select
          blog_entry.title || ': ' || substr(html_strip_tags(blog_entry.body), 0, 1000) as text,
          blog_entry.created
        from
          blog_entry join blog_entry_fts on blog_entry.rowid = blog_entry_fts.rowid
        where
          blog_entry_fts match escape_fts(:q)
        order by
          blog_entry_fts.rank
        limit
          1
        """.strip(),
        "_shape": "array",
        "q": query,
    })
    return response.json()[0]["text"]

In [17]:
def calculate(number):
    return eval(number)

In [18]:
calculate("2+2")

4

In [19]:
known_actions = {
    "wikipedia": wikipedia,
    "calculate": calculate,
    "simon_blog_search": simon_blog_search
}

In [20]:
bot = Chatbot(prompt)

In [21]:
next_prompt="Tell me about quantum computing from Wikipedia."
result = bot(next_prompt)

In [25]:
print(result)

Thought: The user is asking for information about quantum computing from Wikipedia. I should use the wikipedia tool to get this information.
Action: wikipedia: Quantum computing
PAUSE


In [26]:
actions = [action_re.match(a) for a in result.split('\n') if action_re.match(a)]

In [27]:
actions

[<re.Match object; span=(0, 36), match='Action: wikipedia: Quantum computing'>]

In [29]:
action, action_input = actions[0].groups()

In [32]:
action_input

'Quantum computing'

In [33]:
action

'wikipedia'

In [35]:
known_actions[action]

<function wikipedia at 0x000002C70E661EE0>

In [36]:
def query(question,max_turns=5):
    i = 0
    bot = Chatbot(prompt)
    next_prompt = question
    while i < max_turns:
        i += 1
        result = bot(next_prompt)
        print(result)
        actions = [action_re.match(a) for a in result.split('\n') if action_re.match(a)]
        if actions:
            action, action_input = actions[0].groups()
            if action not in known_actions:
                raise Exception(f"Unknown action: {action}: {action_input}")
            print(" -- running {} {}".format(action, action_input))
            observation = known_actions[action](action_input)
            print("Observation:", observation)
            next_prompt = f"Observation: {observation}"
        else:
            return result

In [37]:
query("hello")

Thought: The user said "hello", which is a basic conversational greeting. I should respond with a similar greeting.
Answer: hello


'Thought: The user said "hello", which is a basic conversational greeting. I should respond with a similar greeting.\nAnswer: hello'

In [38]:
print(query("Fifteen * twenty five"))

Thought: The user is asking me to perform a calculation. I should use the calculate tool.
Action: calculate: 15 * 25
PAUSE
 -- running calculate 15 * 25
Observation: 375
Answer: 375
Answer: 375
